In [0]:
%sql
SELECT * FROM ecommerce.bronze.order_items limit 10;

## Loading Libraries for future usage

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


## Load the order_items data and performing quality checks
Validation:
quantity > 0
unit_price >= 0
discount >= 0
discount <= 1

In [0]:
orders_df = (
    spark.read.table("ecommerce.bronze.order_items")
    .withColumn("quantity", col("quantity").cast("decimal(18,2)"))
    .withColumn("unit_price", col("unit_price").cast("decimal(18,2)"))
    .withColumn("discount", col("discount").cast("decimal(18,2)"))
    .dropDuplicates()
    .filter(col("quantity") > 0)
    .filter(col("unit_price") > 0)
    .filter(col("unit_price") >= 0)
    .filter(col("discount") <= 1)
)

## Business transformations 
Calculate:

* gross_amount = quantity × unit_price

* discount_amount = gross_amount × discount

* net_amount = gross_amount - discount_amount

In [0]:
orders_df = orders_df.withColumn('gross_amount',col('quantity')*col('unit_price'))
orders_df = orders_df.withColumn('discount_amount',col('gross_amount')*col('discount'))
orders_df = orders_df.withColumn('net_amount',col('gross_amount')-col('discount_amount'))


In [0]:
orders_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.silver.order_items"
)

In [0]:
%sql

SELECT * FROM ecommerce.SILVER.ORDER_ITEMS LIMIT 10;